In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

Load Data

In [4]:
train_df = pd.read_csv('/content/drive/MyDrive/trainset.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/testset.csv')
print(f"Train: {train_df.shape} | Test: {test_df.shape}")

Train: (127656, 8) | Test: (31915, 8)


Clean Data

In [5]:
def clean_data(df, label_cols):

    df = df.copy()

    # 1. Handle missing/empty comments
    df['comment_text'] = df['comment_text'].fillna('empty').astype(str)
    df['comment_text'] = df['comment_text'].replace('', 'empty')

    # 2. Verify labels are 0 or 1
    for col in label_cols:
        if df[col].isnull().any():
            print(f"Warning: Missing values in {col}, filling with 0")
            df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)
        if not df[col].isin([0, 1]).all():
            print(f"Warning: Invalid values in {col}, setting non-0/1 to 0")
            df[col] = df[col].apply(lambda x: x if x in [0, 1] else 0)

    return df

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
train_df = clean_data(train_df, label_cols)
test_df = clean_data(test_df, label_cols)

print("Data cleaned!")
print(f"Train nulls: {train_df.isnull().sum().sum()}")
print(f"Test nulls: {test_df.isnull().sum().sum()}")

Data cleaned!
Train nulls: 0
Test nulls: 0


Labels

In [6]:
X = train_df['comment_text']
y = train_df[label_cols]
X_test = test_df['comment_text']
y_test = test_df[label_cols]

Train-Validation Split (80/20)

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y.mean(axis=1).round()
)

TF-IDF Vectorization

In [8]:
tfidf = TfidfVectorizer(max_features=50000, stop_words='english', ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

Train Logistic Regression

In [9]:
model = OneVsRestClassifier(LogisticRegression(C=10, max_iter=1000, class_weight='balanced', n_jobs=-1))
model.fit(X_train_tfidf, y_train)
print("Training finished!")

Training finished!


In [10]:
val_pred = model.predict(X_val_tfidf)

Validation Metrics per Category

In [11]:
print("VALIDATION RESULTS \n")
for i, col in enumerate(label_cols):
    tn, fp, fn, tp = confusion_matrix(y_val.iloc[:,i], val_pred[:,i]).ravel()
    f1 = f1_score(y_val.iloc[:,i], val_pred[:,i], zero_division=0)
    print(f"{col:15} → F1: {f1:.4f} | TP:{tp} TN:{tn} FP:{fp} FN:{fn}")

VALIDATION RESULTS 

toxic           → F1: 0.7437 | TP:1985 TN:22179 FP:855 FN:513
severe_toxic    → F1: 0.4746 | TP:196 TN:24902 FP:355 FN:79
obscene         → F1: 0.7885 | TP:1137 TN:23785 FP:392 FN:218
threat          → F1: 0.4914 | TP:57 TN:25357 FP:87 FN:31
insult          → F1: 0.6785 | TP:993 TN:23598 FP:677 FN:264
identity_hate   → F1: 0.4019 | TP:128 TN:25023 FP:307 FN:74


Test Set Evaluation

In [12]:
test_pred = model.predict(X_test_tfidf)

print("\n FINAL TEST SET RESULTS \n")
results = []
for i, col in enumerate(label_cols):
    tn, fp, fn, tp = confusion_matrix(y_test.iloc[:,i], test_pred[:,i]).ravel()
    acc = accuracy_score(y_test.iloc[:,i], test_pred[:,i])
    prec = precision_score(y_test.iloc[:,i], test_pred[:,i], zero_division=0)
    rec = recall_score(y_test.iloc[:,i], test_pred[:,i], zero_division=0)
    f1 = f1_score(y_test.iloc[:,i], test_pred[:,i], zero_division=0)
    results.append({
        'Category': col,
        'F1-Score': f1,
        'Precision': prec,
        'Recall': rec,
        'Accuracy': acc,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    })

results_df = pd.DataFrame(results).sort_values('F1-Score')
display(results_df.round(4))

print("\nEASIEST (highest F1):", results_df.iloc[-2:]['Category'].tolist())
print("HARDEST (lowest F1):", results_df.iloc[:2]['Category'].tolist())


 FINAL TEST SET RESULTS 



,Category,F1-Score,Precision,Recall,Accuracy,TP,TN,FP,FN
3,threat,0.3598,0.2606,0.5811,0.9952,43,31719,122,31
5,identity_hate,0.4312,0.3280,0.6293,0.9847,185,31242,379,109
1,severe_toxic,0.4469,0.3294,0.6947,0.9827,223,31140,454,98
4,insult,0.6887,0.6123,0.7869,0.9640,1270,29497,804,344
0,toxic,0.7446,0.6989,0.7968,0.9477,2435,27810,1049,621
2,obscene,0.7906,0.7424,0.8455,0.9759,1450,29697,503,265



EASIEST (highest F1): ['toxic', 'obscene']
HARDEST (lowest F1): ['threat', 'identity_hate']


In [13]:
#Saving the Predictions
submission = test_df[['id']].copy()
for i, col in enumerate(label_cols):
    submission[col] = test_pred[:, i]
submission.to_csv('/content/drive/MyDrive/logistic_regression_predictions.csv', index=False)
print("Predictions saved to MyDrive!")

Predictions saved to MyDrive!
